# 냉장고를 부탁해 · 식재료 인식 모델 학습 (YOLO26)

Colab 무료 GPU로 식재료 탐지 모델을 학습하고, 브라우저에서 쓰는 ONNX 파일로 내보낸다.

**시작 전에**: 메뉴 `런타임 → 런타임 유형 변경 → T4 GPU`를 선택한다.

순서: ① 환경 준비 → ② 클래스 정의 → ③ 데이터 받기 → ④ 병합 → ⑤ 학습 → ⑥ 평가 → ⑦ 내보내기

## ① 환경 준비

In [ ]:
!nvidia-smi -L
!pip install -q ultralytics roboflow onnx onnxslim onnxruntime

In [ ]:
# 학습 결과를 구글 드라이브에 저장한다. Colab 세션이 끊겨도 결과가 남는다.
from google.colab import drive
drive.mount('/content/drive')

import os
WORK = '/content/drive/MyDrive/fridge_yolo'
os.makedirs(WORK, exist_ok=True)
%cd /content

## ② 클래스 정의

인식할 재료 목록이다. 웹의 `classes.json`과 **같은 파일**을 써야 한다. 재료를 추가하거나 빼면 여기와 웹 양쪽을 같이 바꾼다.

고기, 김치처럼 포장 상태에 따라 모양이 제각각인 재료는 넣지 않는다. 이런 재료는 직접 입력으로 받는다.

In [ ]:
%%writefile classes.json
[
  {"id": 0,  "en": "cabbage",      "ko": "양배추"},
  {"id": 1,  "en": "green_onion",  "ko": "대파"},
  {"id": 2,  "en": "onion",        "ko": "양파"},
  {"id": 3,  "en": "potato",       "ko": "감자"},
  {"id": 4,  "en": "carrot",       "ko": "당근"},
  {"id": 5,  "en": "zucchini",     "ko": "애호박"},
  {"id": 6,  "en": "cucumber",     "ko": "오이"},
  {"id": 7,  "en": "radish",       "ko": "무"},
  {"id": 8,  "en": "spinach",      "ko": "시금치"},
  {"id": 9,  "en": "mushroom",     "ko": "버섯"},
  {"id": 10, "en": "tomato",       "ko": "토마토"},
  {"id": 11, "en": "chili_pepper", "ko": "고추"},
  {"id": 12, "en": "garlic",       "ko": "마늘"},
  {"id": 13, "en": "egg",          "ko": "계란"},
  {"id": 14, "en": "tofu",         "ko": "두부"},
  {"id": 15, "en": "milk",         "ko": "우유"},
  {"id": 16, "en": "apple",        "ko": "사과"},
  {"id": 17, "en": "banana",       "ko": "바나나"},
  {"id": 18, "en": "lemon",        "ko": "레몬"},
  {"id": 19, "en": "bell_pepper",  "ko": "파프리카"}
]

In [ ]:
%%writefile merge_datasets.py
"""
여러 YOLO 형식 데이터셋을 우리 클래스 목록 하나로 합친다.

- 각 데이터셋의 클래스 이름을 우리 클래스(영문 id)로 다시 매핑한다.
- 매핑에 없는 클래스의 박스는 버린다.
- 박스가 하나도 남지 않은 이미지는 '배경 이미지'로 일부만 남긴다(오탐 줄이기용).
- 결과: merged/{train,val,test}/{images,labels} + data.yaml
"""
import json, random, shutil
from pathlib import Path

import yaml

IMG_EXT = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}


def load_classes(path="classes.json"):
    """classes.json → [{'id':0,'en':'cabbage','ko':'양배추'}, ...]"""
    return json.loads(Path(path).read_text(encoding="utf-8"))


def find_split_dirs(root: Path):
    """Roboflow/Label Studio/CVAT YOLO 내보내기의 흔한 폴더 구조를 모두 찾는다."""
    pairs = []
    for img_dir in root.rglob("images"):
        if not img_dir.is_dir():
            continue
        lbl_dir = img_dir.parent / "labels"
        if lbl_dir.is_dir():
            pairs.append((img_dir, lbl_dir))
    return pairs


def source_names(root: Path):
    """data.yaml 또는 classes.txt에서 원본 클래스 이름 목록을 읽는다."""
    y = next(iter(root.rglob("data.yaml")), None)
    if y:
        names = yaml.safe_load(y.read_text(encoding="utf-8"))["names"]
        return [names[k] for k in sorted(names)] if isinstance(names, dict) else list(names)
    t = next(iter(root.rglob("classes.txt")), None)
    if t:
        return [l.strip() for l in t.read_text(encoding="utf-8").splitlines() if l.strip()]
    raise FileNotFoundError(f"{root}: data.yaml / classes.txt 가 없다")


def merge(sources, classes, out="merged", split=(0.7, 0.2, 0.1), keep_background=0.1, seed=42):
    """
    sources: [{'path': 'datasets/roboflow_a', 'map': {'Cabbage': 'cabbage', 'egg': 'egg', ...}}, ...]
             map의 키는 원본 클래스 이름, 값은 우리 클래스 en id
    """
    rnd = random.Random(seed)
    en2id = {c["en"]: c["id"] for c in classes}
    out = Path(out)
    if out.exists():
        shutil.rmtree(out)
    for s in ("train", "val", "test"):
        (out / s / "images").mkdir(parents=True)
        (out / s / "labels").mkdir(parents=True)

    items, counts, dropped = [], {c["en"]: 0 for c in classes}, {}
    for si, src in enumerate(sources):
        root = Path(src["path"])
        names = source_names(root)
        cmap = {}
        for i, n in enumerate(names):
            tgt = src["map"].get(n)
            if tgt is None:
                dropped[n] = dropped.get(n, 0)
                continue
            if tgt not in en2id:
                raise ValueError(f"'{tgt}' 는 classes.json 에 없는 클래스다")
            cmap[i] = en2id[tgt]

        for img_dir, lbl_dir in find_split_dirs(root):
            for img in img_dir.iterdir():
                if img.suffix.lower() not in IMG_EXT:
                    continue
                lbl = lbl_dir / (img.stem + ".txt")
                lines = []
                if lbl.exists():
                    for line in lbl.read_text().splitlines():
                        p = line.split()
                        if len(p) != 5:  # 폴리곤(세그멘테이션) 라벨은 건너뜀
                            continue
                        c = int(p[0])
                        if c in cmap:
                            lines.append(" ".join([str(cmap[c])] + p[1:]))
                            counts[classes[cmap[c]]["en"]] += 1
                        else:
                            n = names[c] if c < len(names) else str(c)
                            dropped[n] = dropped.get(n, 0) + 1
                if not lines and rnd.random() > keep_background:
                    continue
                items.append((img, lines, f"s{si}_{img_dir.parent.name}_{img.stem}"))

    rnd.shuffle(items)
    n = len(items)
    n_tr, n_va = int(n * split[0]), int(n * split[1])
    buckets = {"train": items[:n_tr], "val": items[n_tr:n_tr + n_va], "test": items[n_tr + n_va:]}
    for s, rows in buckets.items():
        for img, lines, stem in rows:
            shutil.copy(img, out / s / "images" / (stem + img.suffix.lower()))
            (out / s / "labels" / (stem + ".txt")).write_text("\n".join(lines))

    data = {
        "path": str(out.resolve()),
        "train": "train/images",
        "val": "val/images",
        "test": "test/images",
        "names": {c["id"]: c["en"] for c in classes},
    }
    (out / "data.yaml").write_text(yaml.safe_dump(data, allow_unicode=True, sort_keys=False), encoding="utf-8")

    print(f"이미지 {n}장 → train {len(buckets['train'])} / val {len(buckets['val'])} / test {len(buckets['test'])}")
    print("클래스별 박스 수:")
    for c in classes:
        k = counts[c["en"]]
        warn = "  ← 부족(최소 150개 권장)" if k < 150 else ""
        print(f"  {c['id']:>2} {c['ko']}({c['en']}): {k}{warn}")
    if dropped:
        print("매핑하지 않아 버린 원본 클래스:", ", ".join(sorted(dropped)))
    return out / "data.yaml"

## ③ 데이터 받기

### 3-1. 공개 데이터셋 (Roboflow Universe)
1. https://universe.roboflow.com 에서 `fridge`, `vegetable`, `ingredient` 등으로 검색한다.
2. 데이터셋 페이지의 **License**를 확인한다. CC BY 4.0, Public Domain 등 사용 가능한 것만 고른다.
3. `Download Dataset → YOLOv8 → show download code`를 누르면 workspace, project, version이 나온다.
4. 아래 `ROBOFLOW_SOURCES`에 적는다. 무료 계정의 API 키가 필요하다(Settings → API Keys).

### 3-2. 우리가 직접 찍은 사진
1. 재료별로 50~100장을 찍는다. 냉장고 칸, 도마 위, 장바구니 등 **실제 쓰는 환경**에서 조명과 각도를 바꿔가며 찍는다.
2. **Label Studio** 또는 **CVAT**(둘 다 무료)로 박스를 친다. 클래스 이름은 `classes.json`의 `en` 값을 그대로 쓴다.
3. YOLO 형식으로 내보낸 zip을 드라이브의 `fridge_yolo/own/` 아래에 풀어둔다.

In [ ]:
from roboflow import Roboflow
from google.colab import userdata

# Colab 왼쪽 열쇠 아이콘(보안 비밀)에 ROBOFLOW_API_KEY 로 저장해 두면 코드에 키가 남지 않는다.
rf = Roboflow(api_key=userdata.get('ROBOFLOW_API_KEY'))

# (workspace, project, version) — 3-1에서 고른 데이터셋으로 바꾼다.
ROBOFLOW_SOURCES = [
    # ('workspace-이름', 'project-이름', 1),
]

downloaded = []
for ws, proj, ver in ROBOFLOW_SOURCES:
    ds = rf.workspace(ws).project(proj).version(ver).download('yolov8', location=f'/content/datasets/{proj}')
    downloaded.append(ds.location)
print(downloaded)

In [ ]:
# 각 데이터셋의 원본 클래스 이름을 확인한다. 다음 셀의 매핑표를 만들 때 쓴다.
import yaml, glob
for d in glob.glob('/content/datasets/*') + glob.glob(f'{WORK}/own/*'):
    for y in glob.glob(f'{d}/**/data.yaml', recursive=True)[:1]:
        names = yaml.safe_load(open(y))['names']
        print(d, '→', list(names.values()) if isinstance(names, dict) else names)

## ④ 병합

데이터셋마다 클래스 이름이 다르다(`Tomato`, `tomatoes`, `tomate`…). 위에서 확인한 원본 이름을 우리 클래스 `en` 값으로 연결한다. **매핑에 없는 클래스의 박스는 버린다.**

In [ ]:
from merge_datasets import load_classes, merge
classes = load_classes('classes.json')

SOURCES = [
    # {'path': '/content/datasets/프로젝트이름', 'map': {'Tomato': 'tomato', 'Egg': 'egg', 'Carrot': 'carrot'}},
    # {'path': f'{WORK}/own/우리사진', 'map': {c['en']: c['en'] for c in classes}},   # 직접 라벨링한 데이터는 이름이 같다
]

DATA_YAML = merge(SOURCES, classes, out='/content/merged')

출력된 **클래스별 박스 수**를 확인한다. 150개 미만인 재료는 사진을 더 찍어서 ④를 다시 실행한다. 데이터가 끝까지 모이지 않는 재료는 ②에서 빼는 게 낫다. 어설프게 인식하는 것보다 직접 입력이 정확하다.

## ⑤ 학습

In [ ]:
from ultralytics import YOLO
from pathlib import Path

model = YOLO('yolo26n.pt')          # 가장 작은 n 모델: 폰 브라우저에서도 돌아간다
model.train(
    data=str(DATA_YAML),
    epochs=100,
    imgsz=640,
    batch=16,
    patience=20,                    # 20 에폭 동안 개선이 없으면 조기 종료
    project=WORK, name='v1', exist_ok=True,
)
BEST = Path(model.trainer.save_dir) / 'weights' / 'best.pt'
print('BEST =', BEST)

In [ ]:
# 세션이 끊겼을 때: 아래 주석을 풀고 실행하면 이어서 학습한다.
# model = YOLO(f'{WORK}/v1/weights/last.pt'); model.train(resume=True)
# BEST = Path(f'{WORK}/v1/weights/best.pt')

## ⑥ 평가

In [ ]:
m = YOLO(str(BEST))
metrics = m.val(data=str(DATA_YAML), split='test')
print(f"전체 mAP50 = {metrics.box.map50:.3f}")
for i, c in enumerate(classes):
    try:
        print(f"  {c['ko']:<6} mAP50 = {metrics.box.class_result(i)[2]:.3f}")
    except Exception:
        print(f"  {c['ko']:<6} (테스트 데이터 없음)")

- 재료별 점수가 낮은 것은 사진을 더 넣는다. `{WORK}/v1/confusion_matrix.png`를 보면 어떤 재료끼리 헷갈리는지 알 수 있다(예: 애호박↔오이).
- 실제 냉장고 사진으로 눈으로 확인하는 게 숫자보다 중요하다.

In [ ]:
from IPython.display import Image, display
import glob
tests = glob.glob(f'{WORK}/test_photos/*.jpg')[:6]   # 학습에 안 쓴 실제 냉장고 사진을 넣어둔다
for r in m.predict(tests, conf=0.35, save=True, project=WORK, name='pred', exist_ok=True):
    print(Path(r.path).name, [(classes[int(b.cls)]['ko'], round(float(b.conf), 2)) for b in r.boxes])
for p in glob.glob(f'{WORK}/pred/*.jpg')[:6]:
    display(Image(p, width=480))

## ⑦ 내보내기 (웹용 ONNX)

`nms=False`로 내보내면 출력이 **[1, 300, 6]**이 된다(`x1, y1, x2, y2, 점수, 클래스`, 640×640 입력 기준 좌표). NMS 후처리가 필요 없어서 브라우저 코드가 단순해진다.

In [ ]:
onnx_path = YOLO(str(BEST)).export(format='onnx', imgsz=640, nms=False)

import onnxruntime as ort, shutil
s = ort.InferenceSession(onnx_path)
print('입력', s.get_inputs()[0].shape, '출력', s.get_outputs()[0].shape)   # [1,3,640,640] / [1,300,6] 이어야 한다
assert s.get_outputs()[0].shape == [1, 300, 6]

shutil.copy(onnx_path, f'{WORK}/fridge.onnx')
shutil.copy('classes.json', f'{WORK}/classes.json')
print('드라이브 fridge_yolo/ 에 fridge.onnx, classes.json 저장 완료')

## 다음 단계
1. 드라이브에서 `fridge.onnx`와 `classes.json`을 받는다.
2. `scan_test.html`을 열어 모델 파일을 고르고 실제 사진으로 확인한다.
3. Spring 프로젝트의 `src/main/resources/static/models/`에 두 파일을 넣는다.